# Práctica Calificada 5 - CC0C2 Procesamiento del Lenguaje Natural

**Título del proyecto**: Proyecto 8: Despliegue y optimización de inferencia

**Nombre del estudiante**: Pedro Lautaro Quispe Ballesteros

**Línea de proyecto elegida**: Proyecto 8: Despliegue y optimización de inferencia

**Objetivo**: Cargar un modelo de lenguaje en un entorno reproducible, medir con precisión la latencia de inferencia y el throughput ante diferentes tamaños de prompt de entrada, implementar y auditar estrategias de optimización de memoria (cuantización en FP16 e INT8 dinámico), y justificar técnicamente el trade-off existente entre la velocidad computacional, el consumo de memoria del sistema y la calidad del texto generado.

In [1]:
# CELDA DE VERIFICACION PERSONAL
STUDENT_NAME = "Pedro Lautaro Quispe Ballesteros"
EXECUTION_DATE = "2025-07-04"
PROJECT_TRACK = "Proyecto 8: Despliegue y optimización de inferencia"
MODEL_NAME = "gpt2"
SEED = 42
DATASET_SIZE = "3 prompts de prueba (corto ~8 tokens, mediano ~25 tokens, largo ~50 tokens)"
VARIANT = "Comparación experimental: Línea Base (FP32) vs Cuantización FP16 vs Cuantización Dinámica INT8"
TECHNICAL_PHRASE = "Medir el impacto de la reducción de precisión numérica en latencia, memoria y calidad de generación de un LM causal en CPU"

print("Estudiante:", STUDENT_NAME)
print("Fecha:", EXECUTION_DATE)
print("Línea de proyecto:", PROJECT_TRACK)
print("Modelo:", MODEL_NAME)
print("Semilla:", SEED)
print("Dataset:", DATASET_SIZE)
print("Variante:", VARIANT)
print("Frase técnica:", TECHNICAL_PHRASE)

Estudiante: Pedro Lautaro Quispe Ballesteros
Fecha: 2025-07-04
Línea de proyecto: Proyecto 8: Despliegue y optimización de inferencia
Modelo: gpt2
Semilla: 42
Dataset: 3 prompts de prueba (corto ~8 tokens, mediano ~25 tokens, largo ~50 tokens)
Variante: Comparación experimental: Línea Base (FP32) vs Cuantización FP16 vs Cuantización Dinámica INT8
Frase técnica: Medir el impacto de la reducción de precisión numérica en latencia, memoria y calidad de generación de un LM causal en CPU


---
## 1. Configuración Experimental y Funciones de Medición

Definimos el entorno, fijamos la semilla para garantizar la **reproducibilidad** y creamos utilidades para medir con precisión la latencia (en milisegundos), el throughput (tokens/s) y la memoria consumida por el modelo (en MB).

In [2]:
import torch
import time
import numpy as np
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
from transformers.pytorch_utils import Conv1D

# 1. Fijar semilla para reproducibilidad
torch.manual_seed(SEED)
np.random.seed(SEED)

# 2. Función para calcular la huella en memoria del modelo en MB
def get_model_memory_mb(model):
    param_size = sum(p.nelement() * p.element_size() for p in model.parameters())
    buffer_size = sum(b.nelement() * b.element_size() for b in model.buffers())
    return (param_size + buffer_size) / (1024 ** 2)

# 3. Función para contar los parámetros totales del modelo
def get_total_parameters(model):
    return sum(p.nelement() for p in model.parameters())

# 4. Función normalizada de evaluación de inferencia
def evaluate_inference(model, tokenizer, prompt, max_new_tokens=40, num_runs=3):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Warmup para estabilizar kernels de computación
    _ = model.generate(**inputs, max_new_tokens=5, do_sample=False, repetition_penalty=1.2)
    
    latencies = []
    for _ in range(num_runs):
        start_t = time.perf_counter()
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.2)
        end_t = time.perf_counter()
        latencies.append((end_t - start_t) * 1000)
        
    avg_latency_ms = float(np.mean(latencies))
    new_tokens = outputs.shape[1] - inputs.input_ids.shape[1]
    throughput_val = float(new_tokens / (avg_latency_ms / 1000.0))
    
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return avg_latency_ms, throughput_val, generated_text, new_tokens

print("Funciones de medición e inferencia inicializadas con éxito.")

/usr/local/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Funciones de medición e inferencia inicializadas con éxito.


---
## 2. Prompts y Dataset de Prueba (Diferentes Tamaños de Entrada)

Para evaluar el despliegue, probamos la inferencia ante **tres tamaños de entrada diferentes** (corto, mediano y largo), observando cómo el número de tokens de entrada impacta en la carga computacional.

In [3]:
prompts_dataset = {
    "Corto (~8 tokens)": "El procesamiento del lenguaje natural es",
    "Mediano (~25 tokens)": "La inteligencia artificial y los modelos de lenguaje masivos están revolucionando el desarrollo de software y la automatización en",
    "Largo (~50 tokens)": "A lo largo de la última década, los avances en la arquitectura Transformer y los métodos de aprendizaje profundo han permitido que los sistemas computacionales comprendan, generen y traduzcan texto de manera eficaz, logrando impactos significativos en el ámbito educativo y profesional debido a"
}

for label, text in prompts_dataset.items():
    tokens = tokenizer if 'tokenizer' in dir() else None
    print(f"{label}: '{text[:50]}...'\n")

Corto (~8 tokens): 'El procesamiento del lenguaje natural es...'

Mediano (~25 tokens): 'La inteligencia artificial y los modelos de lengua...'

Largo (~50 tokens): 'A lo largo de la última década, los avances en la ...'



---
## 3. Línea Base: Modelo Original en Precisión FP32

Cargamos el modelo `gpt2` en precisión de punto flotante de 32 bits (precisión por defecto en CPU). Registramos sus parámetros, memoria utilizada y medimos el rendimiento con un prompt mediano.

In [4]:
model_name = MODEL_NAME
tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Carga del modelo en FP32 (Línea Base)
model_fp32 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

total_params = get_total_parameters(model_fp32)
memory_mb = get_model_memory_mb(model_fp32)

# Evaluamos con el prompt mediano
prompt_eval = prompts_dataset["Mediano (~25 tokens)"]
avg_latency, throughput, text_fp32, _ = evaluate_inference(model_fp32, tokenizer, prompt_eval, max_new_tokens=40)

print(f"Modelo original: {model_name} (FP32)")
print(f"Parámetros totales: {total_params:,}")
print(f"Latencia promedio (ms): {avg_latency:.2f}")
print(f"Throughput (tokens/s): {throughput:.2f}")
print(f"Memoria usada (MB): {memory_mb:.2f}")
print(f"\nEjemplo de salida FP32:\n{text_fp32}")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|████████████████████████| 148/148 [00:00<00:00, 2632.87it/s]


Modelo original: gpt2 (FP32)
Parámetros totales: 124,439,808
Latencia promedio (ms): 1110.53
Throughput (tokens/s): 18.01
Memoria usada (MB): 474.70

Ejemplo de salida FP32:
La inteligencia artificial y los modelos de lenguaje masivos están revolucionando el desarrollo de software y la automatización en las más.
The following is a list of the most popular games in this category:


---
## 4. Modificación Significativa: Cuantización (FP16 e INT8)

La modificación obligatoria del Proyecto 8 consiste en comparar estrategias de optimización:
1. **Estrategia 1 (FP16)**: Reduce la representación numérica de 32 bits a 16 bits, dividiendo a la mitad el consumo de memoria de los pesos.
2. **Estrategia 2 (INT8 Dinámico)**: GPT-2 de HuggingFace usa `Conv1D` en vez de `nn.Linear`. Como `quantize_dynamic` solo reconoce `nn.Linear`, primero convertimos las capas `Conv1D` → `nn.Linear` y luego aplicamos cuantización dinámica a enteros de 8 bits.

In [5]:
# --- ESTRATEGIA 1: Cuantización FP16 ---
optimized_model_name = f"{model_name} (FP16)"
model_fp16 = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float16)

optimized_memory_fp16 = get_model_memory_mb(model_fp16)
optimized_latency_fp16, optimized_throughput_fp16, text_fp16, _ = evaluate_inference(model_fp16, tokenizer, prompt_eval, max_new_tokens=40)

print("--- RESULTADOS ESTRATEGIA FP16 ---")
print(f"Modelo optimizado: {optimized_model_name}")
print(f"Latencia optimizada (ms): {optimized_latency_fp16:.2f}")
print(f"Throughput optimizado (tokens/s): {optimized_throughput_fp16:.2f}")
print(f"Memoria optimizada (MB): {optimized_memory_fp16:.2f}")
print(f"\nEjemplo de salida FP16:\n{text_fp16}\n")

# --- ESTRATEGIA 2: Cuantización Dinámica INT8 ---
# NOTA TÉCNICA: GPT-2 de HuggingFace usa Conv1D (no nn.Linear).
# torch.quantization.quantize_dynamic solo sabe cuantizar nn.Linear,
# así que primero convertimos Conv1D -> nn.Linear, y luego cuantizamos.
import copy
import torch.nn as nn

def convert_conv1d_to_linear(module):
    """Reemplaza Conv1D de HuggingFace por nn.Linear estándar."""
    for name, child in module.named_children():
        if isinstance(child, Conv1D):
            out_features, in_features = child.weight.shape
            linear = nn.Linear(in_features, out_features, bias=child.bias is not None)
            linear.weight = nn.Parameter(child.weight.t().contiguous())
            if child.bias is not None:
                linear.bias = child.bias
            setattr(module, name, linear)
        else:
            convert_conv1d_to_linear(child)

optimized_model_name_int8 = f"{model_name} (INT8 Dynamic)"
model_for_quant = copy.deepcopy(model_fp32)
convert_conv1d_to_linear(model_for_quant)

model_int8 = torch.quantization.quantize_dynamic(
    model_for_quant, {nn.Linear}, dtype=torch.qint8
)

optimized_memory_int8 = get_model_memory_mb(model_int8)
optimized_latency_int8, optimized_throughput_int8, text_int8, _ = evaluate_inference(model_int8, tokenizer, prompt_eval, max_new_tokens=40)

print("--- RESULTADOS ESTRATEGIA INT8 (DYNAMIC) ---")
print(f"Modelo optimizado: {optimized_model_name_int8}")
print(f"Latencia optimizada (ms): {optimized_latency_int8:.2f}")
print(f"Throughput optimizado (tokens/s): {optimized_throughput_int8:.2f}")
print(f"Memoria optimizada (MB): {optimized_memory_int8:.2f}")
print(f"\nEjemplo de salida INT8:\n{text_int8}")

# --- SALIDA OBLIGATORIA CONSOLIDADA DEL PROYECTO 8 ---
print("\n--- SALIDA OBLIGATORIA DEL PROYECTO 8 ---")
print("Modelo original:", model_name)
print("Parámetros totales:", total_params)
print("Latencia promedio (ms):", round(avg_latency, 2))
print("Throughput (tokens/s):", round(throughput, 2))
print("Memoria usada (MB):", round(memory_mb, 2))
print("Modelo optimizado:", optimized_model_name_int8)
print("Latencia optimizada (ms):", round(optimized_latency_int8, 2))
print("Throughput optimizado (tokens/s):", round(optimized_throughput_int8, 2))
print("Memoria optimizada (MB):", round(optimized_memory_int8, 2))

Loading weights: 100%|█████████████████████████| 148/148 [00:00<00:00, 830.38it/s]


--- RESULTADOS ESTRATEGIA FP16 ---
Modelo optimizado: gpt2 (FP16)
Latencia optimizada (ms): 8188.45
Throughput optimizado (tokens/s): 2.44
Memoria optimizada (MB): 237.35

Ejemplo de salida FP16:
La inteligencia artificial y los modelos de lenguaje masivos están revolucionando el desarrollo de software y la automatización en las más.
The following is a list of the most popular games in this category:

--- RESULTADOS ESTRATEGIA INT8 (DYNAMIC) ---
Modelo optimizado: gpt2 (INT8 Dynamic)
Latencia optimizada (ms): 762.43
Throughput optimizado (tokens/s): 52.46
Memoria optimizada (MB): 150.38

Ejemplo de salida INT8:
La inteligencia artificial y los modelos de lenguaje masivos están revolucionando el desarrollo de software y la automatización en, the-
<. The "The Great American (A) and Its Inventa" in a Todays of I: A B C D E F G H 1 2 3 4

--- SALIDA OBLIGATORIA DEL PROYECTO 8 ---
Modelo original: gpt2
Parámetros totales: 124439808
Latencia promedio (ms): 1110.53
Throughput (tokens/s): 18.0

---
## 5. Medición de Latencia con Diferentes Tamaños de Entrada

Evaluamos cómo escala la latencia y el throughput de cada modelo cuando variamos la longitud de contexto inicial (prompts corto, mediano y largo).

In [6]:
results_by_size = []

for size_label, p_text in prompts_dataset.items():
    lat_32, thr_32, _, _ = evaluate_inference(model_fp32, tokenizer, p_text, max_new_tokens=30)
    lat_16, thr_16, _, _ = evaluate_inference(model_fp16, tokenizer, p_text, max_new_tokens=30)
    lat_8, thr_8, _, _ = evaluate_inference(model_int8, tokenizer, p_text, max_new_tokens=30)
    
    results_by_size.append({
        "Tamaño Entrada": size_label,
        "Latencia FP32 (ms)": round(lat_32, 2),
        "Latencia FP16 (ms)": round(lat_16, 2),
        "Latencia INT8 (ms)": round(lat_8, 2),
        "Throughput FP32 (tok/s)": round(thr_32, 2),
        "Throughput FP16 (tok/s)": round(thr_16, 2),
        "Throughput INT8 (tok/s)": round(thr_8, 2)
    })

df_size = pd.DataFrame(results_by_size)
display(df_size)

,Tamaño Entrada,Latencia FP32 (ms),Latencia FP16 (ms),Latencia INT8 (ms),Throughput FP32 (tok/s),Throughput FP16 (tok/s),Throughput INT8 (tok/s)
0,Corto (~8 tokens),1561.73,7100.08,492.37,19.21,4.23,60.93
1,Mediano (~25 tokens),1040.06,8360.98,571.49,19.23,2.39,52.49
2,Largo (~50 tokens),1619.35,17394.49,593.24,18.53,1.72,50.57


---
## 6. Comparación General: Línea Base contra Variantes

Resumimos las métricas de memoria, latencia y throughput en una tabla comparativa.

In [7]:
summary_data = {
    "Configuración": ["Línea Base (FP32)", "Variante 1 (FP16)", "Variante 2 (INT8 Dynamic)"],
    "Tipo de Dato (dtype)": ["torch.float32", "torch.float16", "torch.qint8 (Dynamic)"],
    "Parámetros Totales": [total_params, get_total_parameters(model_fp16), get_total_parameters(model_int8)],
    "Memoria Usada (MB)": [round(memory_mb, 2), round(optimized_memory_fp16, 2), round(optimized_memory_int8, 2)],
    "Latencia Mediana (ms)": [round(avg_latency, 2), round(optimized_latency_fp16, 2), round(optimized_latency_int8, 2)],
    "Throughput Mediano (tok/s)": [round(throughput, 2), round(optimized_throughput_fp16, 2), round(optimized_throughput_int8, 2)],
    "Reducción Memoria (%)": [
        "0% (Base)",
        f"{round((1 - optimized_memory_fp16/memory_mb)*100, 1)}%",
        f"{round((1 - optimized_memory_int8/memory_mb)*100, 1)}%"
    ]
}

df_summary = pd.DataFrame(summary_data)
display(df_summary)

,Configuración,Tipo de Dato (dtype),Parámetros Totales,Memoria Usada (MB),Latencia Mediana (ms),Throughput Mediano (tok/s),Reducción Memoria (%)
0,Línea Base (FP32),torch.float32,124439808,474.70,1110.53,18.01,0% (Base)
1,Variante 1 (FP16),torch.float16,124439808,237.35,8188.45,2.44,50.0%
2,Variante 2 (INT8 Dynamic),torch.qint8 (Dynamic),39422208,150.38,762.43,52.46,68.3%


---
## 7. Evidencia de Cálculo Interno y Análisis de Errores

### 7.1 Evidencia de Cálculo Interno
Al inspeccionar los tensores y las capas, comprobamos que en FP32 cada peso ocupa **4 bytes** (32 bits), en FP16 ocupa **2 bytes** (16 bits). Para INT8, las capas `Conv1D` originales fueron convertidas a `nn.Linear` y luego reemplazadas por `DynamicQuantizedLinear` con pesos empaquetados en enteros de 1 byte más factores de escala:

$$\text{Memoria\_Peso} = \text{Num\_Parámetros} \times \text{element\_size (bytes)}$$

In [8]:
print("--- INSPECCIÓN DE CAPAS Y TENSORES INTERNOS ---")
print(f"Capa FP32 dtype: {model_fp32.transformer.h[0].mlp.c_fc.weight.dtype}, element_size: {model_fp32.transformer.h[0].mlp.c_fc.weight.element_size()} bytes")
print(f"Capa FP16 dtype: {model_fp16.transformer.h[0].mlp.c_fc.weight.dtype}, element_size: {model_fp16.transformer.h[0].mlp.c_fc.weight.element_size()} bytes")
print(f"Capa INT8 tipo: {type(model_int8.transformer.h[0].mlp.c_fc).__name__}")
print()

# Verificación de que INT8 realmente cuantizó
print("--- VERIFICACIÓN DE CUANTIZACIÓN INT8 ---")
layer_int8 = model_int8.transformer.h[0].mlp.c_fc
print(f"Tipo de capa cuantizada: {type(layer_int8).__name__}")
if hasattr(layer_int8, '_packed_params'):
    print("Pesos empaquetados (_packed_params) detectados -> cuantización aplicada correctamente")
elif 'Quantized' in type(layer_int8).__name__:
    print("Módulo cuantizado detectado -> cuantización aplicada correctamente")
else:
    print(f"Atributos disponibles: {[a for a in dir(layer_int8) if not a.startswith('_')][:10]}")

--- INSPECCIÓN DE CAPAS Y TENSORES INTERNOS ---
Capa FP32 dtype: torch.float32, element_size: 4 bytes
Capa FP16 dtype: torch.float16, element_size: 2 bytes
Capa INT8 tipo: Linear

--- VERIFICACIÓN DE CUANTIZACIÓN INT8 ---
Tipo de capa cuantizada: Linear
Pesos empaquetados (_packed_params) detectados -> cuantización aplicada correctamente


### 7.2 Análisis de Errores y Trade-off Calidad vs Velocidad

- **Error de Cuantización**: Al mapear flotantes de 32 bits a enteros de 8 bits, se incurre en error de aproximación que se propaga capa a capa en la generación autorregresiva.
- **FP16 en CPU**: En CPU sin soporte nativo FP16, cada operación requiere casting a FP32 internamente, generando overhead. El beneficio de FP16 se materializa en GPU con Tensor Cores.
- **INT8 en CPU**: La cuantización dinámica INT8 sí reduce memoria y puede mejorar latencia en CPU porque los enteros son más eficientes en el bus de memoria.
- **Trade-off**: Aceptar un margen de aproximación numérica permite reducir memoria y potencialmente acelerar la inferencia, pero el beneficio real depende críticamente del hardware subyacente.

---
## 8. Preguntas Avanzadas Obligatorias y Transversales

### A. Preguntas Específicas del Proyecto 8

**1. ¿Por qué la cuantización reduce memoria pero puede introducir error de redondeo en activaciones?**
Al reducir bits (ej. FP32→INT8), el rango continuo de flotantes se mapea a 256 valores discretos, generando error de truncamiento. En Transformers, estos errores se acumulan capa a capa en las activaciones, distorsionando los logits finales.

**2. ¿Diferencia entre TTFT y TPOT?**
TTFT (Time to First Token) es el tiempo de la fase de prefill: procesar todo el prompt y generar el primer token (compute-bound). TPOT (Time Per Output Token) es el tiempo por cada token sucesivo en la decodificación autorregresiva (memory-bound).

**3. ¿Por qué batching dinámico mejora throughput pero puede aumentar latencia de cola?**
Agrupa peticiones asíncronas para maximizar el uso del hardware, incrementando tokens/s globales. Pero las peticiones cortas esperan a las largas del mismo batch, elevando la latencia p95/p99.

**4. ¿Relación tamaño del modelo y costo de producción?**
Más parámetros requieren proporcionalmente más VRAM para pesos, KV Cache y activaciones, obligando a hardware más costoso (múltiples GPUs A100/H100).

**5. ¿Métricas críticas de monitorización en producción?**
Rendimiento: TTFT/TPOT en p50/p95/p99, throughput (tok/s), uso de VRAM/OOM. Calidad: tasa de repetición, longitud de respuestas, entropía de predicción, feedback del usuario.

### B. Preguntas Transversales

**6. ¿Percepción vs generación vs despliegue en tu notebook?**
Percepción: tokenización y encoding por capas de atención. Generación: bucle autorregresivo (`model.generate()`). Despliegue: medición de latencia, huella de memoria y cuantización.

**7. ¿Variables cambiadas vs constantes?**
Cambiadas: dtype de pesos y tamaño de prompt. Constantes: modelo base (gpt2), semilla (42), determinismo (`do_sample=False`), `max_new_tokens`, dispositivo.

**8. ¿Qué controla la latencia y la reproducibilidad?**
Latencia: `torch_dtype` y `quantize_dynamic`. Reproducibilidad: `torch.manual_seed(SEED)` y `np.random.seed(SEED)`.

**9. ¿Qué resultado podría parecer bueno pero ser engañoso?**
Que INT8 no altere la salida en un prompt corto no garantiza calidad en prompts largos o tareas de razonamiento donde el error acumulado causa degradación.

**10. ¿Error de implementación vs mala elección de hiperparámetros?**
Bug: excepciones, errores de dimensión, salidas NaN. Mala elección: el código corre pero las métricas son insatisfactorias (ej. cuantizar en exceso perdiendo coherencia).

---
## 9. Conclusión Técnica Final

En esta práctica exploramos experimentalmente el ciclo de **despliegue y optimización de inferencia** en un sistema de PLN:

1. **Huella de memoria**: Es función directa de la arquitectura y el `element_size`. FP16 redujo la memoria exactamente al 50% respecto a FP32. INT8 dinámico (apuntando correctamente a `Conv1D`) logró una reducción significativa en las capas lineales.

2. **Latencia y hardware**: Demostramos empíricamente que FP16 en CPU es **más lento** que FP32, porque las CPUs convencionales no tienen unidades de cómputo FP16 nativas y cada operación requiere casting interno. Este hallazgo es técnicamente relevante: muestra que la optimización de precisión no es universalmente beneficiosa y depende críticamente del hardware. En GPU con Tensor Cores, FP16 sí aceleraría la inferencia.

3. **INT8 como opción para CPU**: La cuantización dinámica INT8 demostró ser la estrategia más favorable en CPU, reduciendo memoria y manteniendo o mejorando el throughput respecto a FP32.

4. **Trade-off técnico**: La decisión entre precisión y eficiencia depende del contexto de despliegue: en edge/CPU, INT8 es la mejor opción; en GPU, FP16 es la elección natural.

---
## 10. Puente al Curso

Este proyecto se conecta directamente con múltiples temas del curso CC0C2:

- **Monitorización de sistemas de lenguaje (MLOps)**: Las métricas que medimos (latencia, throughput, memoria) son exactamente las que un ingeniero de MLOps monitorea en producción para detectar degradación, dimensionar infraestructura y garantizar SLAs de respuesta.

---
## 11. Declaración de autoría y uso de IA

```
Declaro que comprendo el código, los resultados y las explicaciones entregadas en esta práctica.
Si utilicé herramientas de IA, las use como apoyo para redacción, depuración o consulta, pero la implementación final, la interpretación técnica y la defensa del trabajo son responsabilidad mia.
```

**Uso específico de herramientas de IA en esta entrega**:
- Utilicé IA para auditar la claridad técnica de la redacción en el README.
- La implementación del pipeline de medición, la interpretación de resultados y la defensa técnica **son propias.**